In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("Real estate.csv")

In [3]:
df

,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,1,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.917,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833,5.0,390.56840,5,24.97937,121.54245,43.1
...,...,...,...,...,...,...,...,...
409,410,2013.000,13.7,4082.01500,0,24.94155,121.50381,15.4
410,411,2012.667,5.6,90.45606,9,24.97433,121.54310,50.0
411,412,2013.250,18.8,390.96960,7,24.97923,121.53986,40.6
412,413,2013.000,8.1,104.81010,5,24.96674,121.54067,52.5


In [4]:
df = df.drop(columns=["No"])

In [5]:
df

,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
1,2012.917,19.5,306.59470,9,24.98034,121.53951,42.2
2,2013.583,13.3,561.98450,5,24.98746,121.54391,47.3
3,2013.500,13.3,561.98450,5,24.98746,121.54391,54.8
4,2012.833,5.0,390.56840,5,24.97937,121.54245,43.1
...,...,...,...,...,...,...,...
409,2013.000,13.7,4082.01500,0,24.94155,121.50381,15.4
410,2012.667,5.6,90.45606,9,24.97433,121.54310,50.0
411,2013.250,18.8,390.96960,7,24.97923,121.53986,40.6
412,2013.000,8.1,104.81010,5,24.96674,121.54067,52.5


In [13]:
X=df.iloc[:,:-1].values

In [14]:
X

array([[2012.917  ,   32.     ,   84.87882,   10.     ,   24.98298,
         121.54024],
       [2012.917  ,   19.5    ,  306.5947 ,    9.     ,   24.98034,
         121.53951],
       [2013.583  ,   13.3    ,  561.9845 ,    5.     ,   24.98746,
         121.54391],
       ...,
       [2013.25   ,   18.8    ,  390.9696 ,    7.     ,   24.97923,
         121.53986],
       [2013.     ,    8.1    ,  104.8101 ,    5.     ,   24.96674,
         121.54067],
       [2013.5    ,    6.5    ,   90.45606,    9.     ,   24.97433,
         121.5431 ]], shape=(414, 6))

In [15]:
y=df.iloc[:,-1].values.reshape(-1,1)

In [17]:
print(X.shape, y.shape)

(414, 6) (414, 1)


In [24]:
def generate_data(n=100):
    X = np.random.rand(n, 1) * 10
    y = 3 * X + 5 + np.random.randn(n, 1)
    return X, y


In [25]:
def standardize(X):
    mean = X.mean(axis=0)
    std = X.std(axis=0) + 1e-8
    X_scaled = (X - mean) / std
    return X_scaled, mean, std


In [26]:
def mse(y, y_pred):
    return np.mean((y - y_pred) ** 2)


In [27]:
def fit_batch(X, y, lr=0.01, epochs=1000):
    n, d = X.shape
    Xb = np.c_[np.ones((n, 1)), X]  # bias    The code Xb = np.c_[np.ones((n,1)), X] is a common NumPy operation used in machine learning (specifically linear regression) to add a bias column (column of ones) to a feature matrix \(X\). Breakdown of the Code n: Represents the number of samples (rows) in your dataset.np.ones((n, 1)): Creates a new column vector of shape (\(n\times 1\)) filled with the value \(1.0\).np.c_[...]: Stands for column-wise concatenation. It takes the column of ones and attaches the original matrix \(X\) to its right.Xb: The resulting matrix, which now has shape (\(n\times (m+1)\)), where \(m\) is the original number of features. Purpose In linear regression, the model is defined as \(y=\theta _{0}+\theta _{1}x_{1}+\theta _{2}x_{2}+\dots +\theta _{m}x_{m}\).To express this using matrix multiplication (\(y=Xb\cdot \theta \)), a dummy variable \(x_{0}=1\) is needed for the intercept term \(\theta _{0}\). Adding this column allows the formula to include the intercept directly in the matrix math
    w = np.zeros((d+1, 1))

    losses = []

    for _ in range(epochs):
        y_pred = Xb @ w
        error = y_pred - y

        grad = (2/n) * Xb.T @ error
        w -= lr * grad

        losses.append(mse(y, y_pred))

    return w, losses


In [22]:
def fit_sgd(X, y, lr=0.01, epochs=50):
    n, d = X.shape
    Xb = np.c_[np.ones((n, 1)), X]
    w = np.zeros((d+1, 1))

    losses = []

    for _ in range(epochs):
        for i in range(n):
            xi = Xb[i:i+1]
            yi = y[i:i+1]

            error = xi @ w - yi
            grad = 2 * xi.T @ error
            w -= lr * grad

        losses.append(mse(y, Xb @ w))

    return w, losses


In [23]:
def fit_minibatch(X, y, lr=0.01, batch_size=32, epochs=200):
    n, d = X.shape
    Xb = np.c_[np.ones((n, 1)), X]
    w = np.zeros((d+1, 1))
    losses = []

    for _ in range(epochs):
        idx = np.random.permutation(n)
        Xb_shuff = Xb[idx]
        y_shuff = y[idx]

        for i in range(0, n, batch_size):
            xb = Xb_shuff[i:i+batch_size]
            yb = y_shuff[i:i+batch_size]

            grad = (2/len(xb)) * xb.T @ (xb @ w - yb)
            w -= lr * grad

        losses.append(mse(y, Xb @ w))

    return w, losses
